# 03 — Deep models on a GPU (option C)

Trains **N-HiTS** and **TFT** (neuralforecast) for each backtest origin and writes their forecasts
next to the inputs, so they can be scored against LightGBM with `idf-deep score`.

Inputs come from `idf-deep export` (`data/deep_colab/inputs/`): one file per origin with the
history (target `log1p(entries)`, `available_mask` = 0 on excluded days) and the future
exogenous variables, already built with the backtest's no-leakage rules. This notebook only
trains and predicts.

**Steps**
1. Put the folder `navigo_deep` (with `inputs/` inside) in *My Drive*, or just the file
   `navigo_deep.zip` at the top of *My Drive* (it is unpacked on the first run).
2. Runtime → Change runtime type → **L4 GPU** (T4 works; A100 is faster for TFT). Not TPU.
3. Runtime → Run all. Each model × origin is saved as soon as it is done; after a disconnect,
   run all again: finished ones are skipped.
4. Download `My Drive/navigo_deep/outputs.zip` (made by the last cell) and unzip it into
   `idf-crowding/data/deep_colab/outputs/`, then run `make deep-score`.

**Runs** (Settings cell): each run = model × seed × set of origins (`scored` = the 12
backtest origins, `warmup` = the 12 origins before them, which only feed the forecast log
used to calibrate the ensemble's intervals). Seed 1 keeps the plain model name (`tft`), so
files from earlier runs are reused; other seeds are saved as `tft_s2`, `tft_s3`, …

To split the runs across parallel Colab sessions (Colab Pro), set `PART` = 1, 2, 3 in the
Settings cell of each session.

A newer `navigo_deep.zip` at the top of *My Drive* is unpacked automatically (inputs only;
`outputs/` and this notebook are never overwritten).

Expected time on an L4: TFT ~5–15 min per origin, N-HiTS ~2–5 min.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > L4 GPU"
# 3.3.0 keeps Colab's pandas 2.2.3 (3.3.1 pulls pandas 3, which breaks google-colab)
%pip install -q neuralforecast==3.3.0 pandas==2.2.3

In [ ]:
import json
import shutil
import time
import traceback
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
BASE = DRIVE / "navigo_deep"
ZIP = DRIVE / "navigo_deep.zip"
META_PATH = BASE / "inputs" / "meta.json"
if ZIP.exists():
    with zipfile.ZipFile(ZIP) as z:
        new_meta = json.loads(z.read("navigo_deep/inputs/meta.json"))
        old_meta = json.loads(META_PATH.read_text()) if META_PATH.exists() else None
        if new_meta != old_meta:  # unpack inputs only: outputs/ and the notebook stay as they are
            members = [m for m in z.namelist() if m.startswith("navigo_deep/inputs/")]
            z.extractall(DRIVE, members=members)
            print(f"unpacked {len(members)} input files from {ZIP.name}")
assert META_PATH.exists(), f"inputs not found under {BASE}"
print(sorted(p.name for p in (BASE / "inputs").glob("*.parquet") if "_static" not in p.name))

In [ ]:
# Settings. One run = (model, seed, origins); origins = "scored" or "warmup".
# Seed 1 keeps the plain name ("tft"): files from earlier runs are reused, not recomputed.
RUNS = [
    ("tft", 2, "scored"),  # seed robustness
    ("tft", 1, "warmup"),  # forecast log -> calibrated intervals for mean(lgbm, tft)
    ("tft", 3, "scored"),  # seed robustness
]
# Parallel sessions: PART = 1, 2 or 3 runs only RUNS[PART - 1]; None runs all of them.
# Start part 1 first and wait for "unpacked ..." (or the file list) before starting the others.
PART = None
QUICK_TEST = False  # True: 12 stations, 6 steps, to check the setup in a minute

In [ ]:
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MQLoss
from neuralforecast.models import NHITS, TFT

INP, OUT = BASE / "inputs", BASE / ("outputs_test" if QUICK_TEST else "outputs")
OUT.mkdir(parents=True, exist_ok=True)
META = json.loads((INP / "meta.json").read_text())
H, FUTR, STAT, QS = META["horizon"], META["futr_exog"], META["stat_exog"], META["quantiles"]
ORIGINS = {
    role: [o["origin"] for o in META["origins"] if o.get("role", "scored") == role]
    for role in ("warmup", "scored")
}
print({k: len(v) for k, v in ORIGINS.items()})
GPU = torch.cuda.is_available()
torch.set_float32_matmul_precision("high")

CONFIGS = {
    "nhits": dict(
        cls=NHITS,
        input_size=182,
        max_steps=3000,
        learning_rate=1e-3,
        batch_size=128,
        windows_batch_size=1024,
        scaler_type="robust",
        early_stop_patience_steps=5,
        val_check_steps=100,
        random_seed=1,
    ),
    "tft": dict(
        cls=TFT,
        input_size=182,
        hidden_size=64,
        n_head=4,
        max_steps=2000,
        learning_rate=1e-3,
        batch_size=64,
        windows_batch_size=128,
        scaler_type="robust",
        early_stop_patience_steps=5,
        val_check_steps=100,
        random_seed=1,
    ),
}


def run_name(model, seed):
    return model if seed == 1 else f"{model}_s{seed}"


def build(model, seed):
    c = dict(CONFIGS[model])
    cls = c.pop("cls")
    c["random_seed"] = seed
    if QUICK_TEST:
        c.update(max_steps=6, val_check_steps=3)
    return cls(
        h=H,
        futr_exog_list=FUTR,
        stat_exog_list=STAT,
        loss=MQLoss(quantiles=QS),
        valid_loss=MQLoss(quantiles=QS),
        alias=run_name(model, seed),
        accelerator="gpu" if GPU else "cpu",
        devices=1,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        **c,
    )


def pick(columns, name, q):
    """Column of quantile q in the MQLoss output (NAME-lo-80.0 / NAME-median / NAME-hi-80.0)."""
    pref = f"{name}-median" if q == 0.5 else f"{name}-lo-" if q < 0.5 else f"{name}-hi-"
    hits = [c for c in columns if c.startswith(pref)]
    if len(hits) != 1:
        raise KeyError(f"no unique column for {name} q={q} in {list(columns)}")
    return hits[0]


def run_origin(model, seed, origin):
    name = run_name(model, seed)
    out = OUT / f"{name}_{origin}.parquet"
    if out.exists():
        print(f"{name} {origin}: done already")
        return
    t0 = time.time()
    df = pd.read_parquet(INP / f"{origin}.parquet")
    static = pd.read_parquet(INP / f"{origin}_static.parquet")
    if QUICK_TEST:
        ids = static["unique_id"].iloc[:12]
        df, static = df[df.unique_id.isin(ids)], static[static.unique_id.isin(ids)]
    df["ds"] = pd.to_datetime(df["ds"])
    hist = df[~df.is_future][["unique_id", "ds", "y", "available_mask", *FUTR]]
    futr = df[df.is_future][["unique_id", "ds", *FUTR]]
    nf = NeuralForecast(models=[build(model, seed)], freq="D")
    nf.fit(df=hist, static_df=static, val_size=H)  # validation = last 60 days before the origin
    fc = nf.predict(futr_df=futr)
    fc = fc if "unique_id" in fc.columns else fc.reset_index()
    res = pd.DataFrame(
        {
            "station_id": fc["unique_id"].astype("int64"),
            "date": pd.to_datetime(fc["ds"]).dt.date,
            "origin": pd.Timestamp(origin).date(),
            "model": name,
        }
    )
    for q in QS:  # back from log1p to entries
        res[f"q{int(round(q * 100))}"] = np.expm1(fc[pick(fc.columns, name, q)].to_numpy()).clip(
            min=0
        )
    assert len(res) == len(static) * H, (len(res), len(static))
    res.to_parquet(out, index=False)
    print(f"{name} {origin}: {len(static)} series, {time.time() - t0:.0f} s")

In [ ]:
THIS_RUNS = RUNS if PART is None else [RUNS[PART - 1]]
print("this session:", THIS_RUNS)
for model, seed, role in THIS_RUNS:
    for origin in ORIGINS[role]:
        try:
            run_origin(model, seed, origin)
        except Exception:
            print(f"{run_name(model, seed)} {origin}: FAILED")
            traceback.print_exc()
        finally:
            torch.cuda.empty_cache() if GPU else None

In [ ]:
done = sorted(p.name for p in OUT.glob("*.parquet"))
expected = [f"{run_name(m, s)}_{o}.parquet" for m, s, r in RUNS for o in ORIGINS[r]]
missing = [f for f in expected if f not in done]
mine = [f"{run_name(m, s)}_{o}.parquet" for m, s, r in THIS_RUNS for o in ORIGINS[r]]
print(f"this session: {'done' if not set(mine) & set(missing) else 'MISSING files'}")
print(f"{len(done)} files in outputs; missing for RUNS: {missing or 'none'}")
archive = shutil.make_archive(str(BASE / OUT.name), "zip", OUT)
print("download:", archive)